# Seven-Level Inverter: CNN Waveform Baseline

This notebook imports the project backend; it does not define a second model implementation. The prediction task uses 400-point input voltage and load-current waveforms plus four operating conditions to predict the dataset's 400-point target/reference voltage. The provided split IDs are preserved. Validation is used for model selection; the test split is evaluated only after training.

The ZIP reports **7 voltage levels**, 3 H-bridge cells, 12 switches, a 50 Hz fundamental and 20 kHz sampling. Gate/state channels have separate dimensions and are not CNN outputs. Reference/target THD is not a measurement of post-inverter output THD.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Experiment configuration

Upload the complete `Hybrid_CNN_Inverter_Optimization` project directory, including backend modules, to Drive. Edit the two paths below if your Drive layout differs. The checkpoint/run directory should remain on Drive so it persists after the Colab session.

In [ ]:
from pathlib import Path
from datetime import datetime
import os, sys, json, yaml, torch

PROJECT_ROOT = Path('/content/drive/MyDrive/Hybrid_CNN_Inverter_Optimization')
DATASET_ZIP = PROJECT_ROOT / 'data/raw/Seven_Level_Inverter_Project.zip'
DATASET_EXTRACTED = PROJECT_ROOT / 'data/raw/Seven_Level_Inverter_Project_extracted'
RUN_NAME = 'baseline_' + datetime.now().strftime('%Y%m%d_%H%M%S')
EXPERIMENT_DIR = PROJECT_ROOT / 'colab_runs' / RUN_NAME
RESUME = False  # Set True and point EXPERIMENT_DIR at an existing run to resume it.
CONFIG_PATH = PROJECT_ROOT / 'configs/config.yaml'

assert PROJECT_ROOT.is_dir(), f'Project folder not found: {PROJECT_ROOT}'
assert DATASET_ZIP.is_file(), f'Dataset ZIP not found: {DATASET_ZIP}'
sys.path.insert(0, str(PROJECT_ROOT))
print('Project:', PROJECT_ROOT)
print('Dataset:', DATASET_ZIP)
print('GPU available:', torch.cuda.is_available())

## Install project dependencies

Colab usually already provides PyTorch. This cell installs the project's declared dependencies and does not start training.

In [ ]:
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements.txt')], check=True)

## Imports and reproducible settings

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from preprocessing.data_loader import load_dataset
from preprocessing.data_validation import validate_dataset
from preprocessing.preprocess import preprocess_dataset
from training.dataset import create_dataloaders
from training.trainer import TrainConfig, Trainer, set_reproducibility
from models.model_factory import create_model
from models.cnn_model import detect_device
from evaluation.evaluate import evaluate_model, plot_training_history, plot_waveform_comparison, plot_harmonic_spectrum, plot_thd_comparison
from evaluation.thd_analysis import calculate_thd

with CONFIG_PATH.open(encoding='utf-8') as file:
    PROJECT_CONFIG = yaml.safe_load(file)
assert PROJECT_CONFIG['dataset']['waveform_points'] == 400
assert PROJECT_CONFIG['dataset']['sampling_frequency_hz'] == 20000
assert PROJECT_CONFIG['dataset']['fundamental_frequency_hz'] == 50
DEVICE = detect_device()
print('Device:', DEVICE)

## Extract and validate the supplied dataset

The loader extracts the known CSVs from the ZIP. Validation checks required columns, values, SampleID coverage, configured voltage levels, split counts, and the large auxiliary files in chunks. No random split is created.

In [ ]:
TABLES = load_dataset(DATASET_ZIP, extract_to=DATASET_EXTRACTED)
VALIDATION = validate_dataset(TABLES, report_path=EXPERIMENT_DIR / 'dataset_validation.json')
print(json.dumps(VALIDATION.summary, indent=2)[:12000])
for warning in VALIDATION.warnings:
    print('WARNING:', warning)

## Preprocess with training-only normalization

All tables are aligned by SampleID. Waveform and condition normalization statistics are fitted on training records only, then reused for validation and testing.

In [ ]:
PREPARED = preprocess_dataset(TABLES, metadata_path=EXPERIMENT_DIR / 'preprocessing_metadata.json')
print(PREPARED.metadata())
print('Split indices:', {name: len(PREPARED.indices(name)) for name in ['training', 'validation', 'testing']})

## Inspect a dataset example

This is a real sample selected from the supplied data. Waveform values below are converted back to engineering units.

In [ ]:
example_index = 0
time_ms = np.arange(400) / PROJECT_CONFIG['dataset']['sampling_frequency_hz'] * 1000
input_v = PREPARED.input_voltage[example_index] * PREPARED.stats.input_voltage_std + PREPARED.stats.input_voltage_mean
target_v = PREPARED.stats.denormalize_target(PREPARED.target_voltage[example_index])
load_i = PREPARED.load_current[example_index] * PREPARED.stats.load_current_std + PREPARED.stats.load_current_mean
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(time_ms, input_v); axes[0].plot(time_ms, target_v); axes[0].set_ylabel('Voltage (V)'); axes[0].legend(['Input inverter voltage', 'Target/reference voltage'])
axes[1].plot(time_ms, load_i); axes[1].set_ylabel('Current (A)'); axes[1].set_xlabel('Time (ms)')
fig.suptitle(f'SampleID {PREPARED.sample_ids[example_index]} | split {PREPARED.splits[example_index]}')
plt.tight_layout()

## PyTorch datasets and loaders

The training loop receives only the training and validation loaders. A distinct test loader is held until final evaluation.

In [ ]:
TRAINING = PROJECT_CONFIG['training']
set_reproducibility(int(TRAINING['seed']))
LOADERS = create_dataloaders(
    PREPARED,
    batch_size=int(TRAINING['batch_size']),
    num_workers=0,
    seed=int(TRAINING['seed']),
    pin_memory=(DEVICE.type == 'cuda'),
)
batch = next(iter(LOADERS.train))
{key: tuple(value.shape) for key, value in batch.items()}

## CNN architecture and initialization

The backend's configurable residual CNN has independent voltage/current feature branches and an operating-condition encoder. Its output is a 400-point normalized waveform.

In [ ]:
MODEL = create_model(PROJECT_CONFIG)
MODEL = MODEL.to(DEVICE)
MODEL.eval()
with torch.no_grad():
    example_output = MODEL(
        batch['input_voltage'].to(DEVICE),
        batch['load_current'].to(DEVICE),
        batch['operating_conditions'].to(DEVICE),
    )
print('Output shape:', tuple(example_output.shape))
print('Trainable parameters:', MODEL.parameter_count())

## Editable training configuration and Drive checkpointing

The baseline uses AdamW and MAE with an optional derivative penalty. Early stopping/checkpoint selection use validation loss only. Set `RESUME = True` and `EXPERIMENT_DIR` to an existing run directory to continue its saved optimizer/scheduler state.

In [ ]:
TRAIN_CONFIG = TrainConfig(
    epochs=int(TRAINING['epochs']),
    learning_rate=float(TRAINING['learning_rate']),
    weight_decay=float(TRAINING['weight_decay']),
    patience=int(TRAINING['patience']),
    min_delta=float(TRAINING['min_delta']),
    loss=str(TRAINING['loss']),
    derivative_weight=float(TRAINING['derivative_weight']),
    scheduler_patience=int(TRAINING['scheduler_patience']),
    scheduler_factor=float(TRAINING['scheduler_factor']),
    minimum_learning_rate=float(TRAINING['minimum_learning_rate']),
    gradient_clip_norm=TRAINING.get('gradient_clip_norm'),
    seed=int(TRAINING['seed']),
)
CHECKPOINT_DIR = EXPERIMENT_DIR / 'checkpoints'
RESULTS_DIR = EXPERIMENT_DIR / 'results'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / 'best.pt'
MODEL_METADATA = {
    'model_config': PROJECT_CONFIG['model'],
    'normalization': PREPARED.metadata()['normalization'],
    'operating_feature_names': PREPARED.metadata()['operating_feature_names'],
}
TRAINER = Trainer(
    MODEL,
    TRAIN_CONFIG,
    device=DEVICE,
    checkpoint_path=CHECKPOINT_PATH,
    history_path=RESULTS_DIR / 'training_history.csv',
    checkpoint_metadata=MODEL_METADATA,
)
if RESUME:
    assert CHECKPOINT_PATH.is_file(), f'No checkpoint to resume: {CHECKPOINT_PATH}'
    TRAINER.resume()
print('Checkpoint:', CHECKPOINT_PATH)

## Training execution

Run this cell only when ready to train. It does not access the test loader. Training can take a long time depending on the selected runtime/GPU.

In [ ]:
HISTORY = TRAINER.fit(LOADERS.train, LOADERS.validation, resume=False)
print(f'Completed {len(HISTORY)} epochs. Best validation loss: {TRAINER.best_validation_loss:.6g}')
assert CHECKPOINT_PATH.is_file()

## Training and validation loss

In [ ]:
fig = plot_training_history(HISTORY, RESULTS_DIR / 'training_validation_loss.png')
plt.show()

## Final held-out test evaluation

This is the first point at which the test loader is used. Do not use these results to revise hyperparameters/checkpoint selection.

In [ ]:
from training.checkpoint import load_checkpoint
best_state = load_checkpoint(CHECKPOINT_PATH, map_location=DEVICE)
MODEL.load_state_dict(best_state['model_state'])
TEST_RESULT = evaluate_model(MODEL, LOADERS.test, device=DEVICE, normalization=PREPARED.stats)
print(json.dumps(TEST_RESULT['metrics'], indent=2))

## Prediction versus reference waveform

In [ ]:
fig = plot_waveform_comparison(
    TEST_RESULT['targets'], TEST_RESULT['predictions'], sample_index=0,
    sampling_frequency_hz=PROJECT_CONFIG['dataset']['sampling_frequency_hz'],
    path=RESULTS_DIR / 'test_waveform_comparison.png',
)
plt.show()
print('Displayed sampleID:', int(TEST_RESULT['sample_ids'][0]))

## FFT and explicitly labeled waveform THD

The values below compare the dataset target/reference and the CNN's predicted target waveform on one held-out sample. The prediction is **not** a measured inverter output. Neither this calculation nor the CSV's `targetVoltageTHD` establishes physical post-inverter THD.

In [ ]:
test_reference = TEST_RESULT['targets'][0]
test_prediction = TEST_RESULT['predictions'][0]
fs = float(PROJECT_CONFIG['dataset']['sampling_frequency_hz'])
f0 = float(PROJECT_CONFIG['dataset']['fundamental_frequency_hz'])
max_harmonic = int(PROJECT_CONFIG['dataset']['max_harmonic'])
reference_thd = calculate_thd(test_reference, sampling_frequency_hz=fs, fundamental_frequency_hz=f0, max_harmonic=max_harmonic)
prediction_thd = calculate_thd(test_prediction, sampling_frequency_hz=fs, fundamental_frequency_hz=f0, max_harmonic=max_harmonic)
print({'dataset_target_reference_waveform_thd_percent': reference_thd,
       'cnn_predicted_target_waveform_thd_percent': prediction_thd,
       'interpretation': 'waveform FFT only; not measured inverter output THD'})
fig = plot_harmonic_spectrum(test_prediction, sampling_frequency_hz=fs,
    fundamental_frequency_hz=f0, max_harmonic=max_harmonic,
    path=RESULTS_DIR / 'predicted_target_harmonics.png')
plt.show()
fig = plot_thd_comparison(test_reference, test_prediction, sampling_frequency_hz=fs,
    fundamental_frequency_hz=f0, max_harmonic=max_harmonic,
    path=RESULTS_DIR / 'target_prediction_thd_comparison.png')
plt.show()

## Export results to Google Drive

The validation report, preprocessing metadata, training history, best checkpoint, and final test metrics are kept under the selected experiment directory. No metric is generated unless the preceding training/evaluation cells are run.

In [ ]:
summary = {
    'experiment_dir': str(EXPERIMENT_DIR),
    'dataset_sample_id': int(TEST_RESULT['sample_ids'][0]),
    'test_metrics': TEST_RESULT['metrics'],
    'reference_target_waveform_thd_percent': float(reference_thd),
    'cnn_prediction_waveform_thd_percent': float(prediction_thd),
    'thd_interpretation': 'CNN predicted target waveform; not measured inverter output THD',
    'sampling_frequency_hz': fs,
    'fundamental_frequency_hz': f0,
    'checkpoint': str(CHECKPOINT_PATH),
}
(RESULTS_DIR / 'test_metrics.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Saved:', RESULTS_DIR / 'test_metrics.json')